# Analyse des ventes d'une PME

Ce notebook explore l'extrait de 20 jours de ventes fourni par le client (`ventes.csv`) :

| Colonne   | Description                   |
|-----------|-------------------------------|
| `date`    | date de la vente (AAAA-MM-JJ) |
| `produit` | nom du produit                |
| `prix`    | prix unitaire                 |
| `qte`     | quantité vendue               |
| `region`  | région de la vente            |

Le **chiffre d'affaires** d'une vente est calculé comme `prix × qte`.

**Plan**

1. Exploration du jeu de données avec Pandas : moyenne, médiane, écart-type, variance
2. Produit le plus vendu et le moins vendu, en Python natif (sans Pandas)
3. Graphiques avec Plotly Express : ventes par produit, chiffre d'affaires par produit

Prérequis : `uv sync`, puis `uv run jupyter notebook` (voir le README).

## Chargement des données

In [1]:
import pandas as pd
import plotly.express as px

# Chargement du fichier CSV fourni par le client
ventes = pd.read_csv("ventes.csv")

# Le chiffre d'affaires d'une vente = prix unitaire × quantité vendue
ventes["chiffre_affaires"] = ventes["prix"] * ventes["qte"]

print(f"{len(ventes)} ventes chargées")
ventes.head()

39 ventes chargées


,date,produit,prix,qte,region,chiffre_affaires
0,2022-01-01,Produit A,10,100,Nord,1000
1,2022-01-02,Produit B,15,50,Nord,750
2,2022-01-02,Produit A,10,75,Sud,750
3,2022-01-03,Produit C,20,30,Nord,600
4,2022-01-03,Produit A,10,150,Sud,1500


In [2]:
# Aperçu statistique des colonnes numériques
ventes.describe()

,prix,qte,chiffre_affaires
count,39.000000,39.000000,39.000000
mean,14.743590,86.666667,1149.358974
std,4.127685,41.916793,368.160755
min,10.000000,25.000000,500.000000
25%,10.000000,55.000000,800.000000
50%,15.000000,80.000000,1200.000000
75%,20.000000,110.000000,1500.000000
max,20.000000,200.000000,2000.000000


## 1. Exploration avec Pandas

### 1.a Chiffre d'affaires et volume des ventes par produit : moyenne et médiane

Une ligne du CSV correspond à une vente. Les statistiques ci-dessous sont donc calculées,
pour chaque produit, sur le chiffre d'affaires et le volume (quantité) de ses ventes.

In [3]:
# Moyenne et médiane, par produit, du chiffre d'affaires et du volume de chaque vente
statistiques_par_produit = ventes.groupby("produit").agg(
    ca_moyenne=("chiffre_affaires", "mean"),
    ca_mediane=("chiffre_affaires", "median"),
    volume_moyenne=("qte", "mean"),
    volume_mediane=("qte", "median"),
)
statistiques_par_produit.round(2)

,ca_moyenne,ca_mediane,volume_moyenne,volume_mediane
produit,,,,
Produit A,1250.00,1225.0,125.00,122.5
Produit B,1217.31,1200.0,81.15,80.0
Produit C,958.33,900.0,47.92,45.0


### 1.b Volume des ventes par produit : écart-type et variance

In [4]:
# Dispersion du volume des ventes par produit.
# Pandas utilise par défaut l'estimateur non biaisé (ddof=1) : division par n - 1.
dispersion_volume_par_produit = ventes.groupby("produit")["qte"].agg(
    ecart_type="std",
    variance="var",
)
dispersion_volume_par_produit.round(2)

,ecart_type,variance
produit,,
Produit A,38.08,1450.00
Produit B,21.23,450.64
Produit C,17.90,320.27


## 2. Produit le plus vendu et le moins vendu (Python natif, sans Pandas)

Le fichier CSV est relu avec le module standard `csv`. Les unités vendues sont cumulées
par produit dans un dictionnaire, puis `max()` et `min()` désignent les deux extrêmes.

In [5]:
import csv

# Cumul des unités vendues par produit : {nom du produit: total d'unités}
unites_par_produit = {}
with open("ventes.csv", newline="", encoding="utf-8") as fichier:
    for vente in csv.DictReader(fichier):
        produit = vente["produit"]
        unites_par_produit[produit] = unites_par_produit.get(produit, 0) + int(vente["qte"])

# max() et min() comparent les produits selon leur total d'unités (key=unites_par_produit.get)
produit_plus_vendu = max(unites_par_produit, key=unites_par_produit.get)
produit_moins_vendu = min(unites_par_produit, key=unites_par_produit.get)

print("Unités vendues par produit :", unites_par_produit)
print(f"Produit le plus vendu  : {produit_plus_vendu} ({unites_par_produit[produit_plus_vendu]} unités)")
print(f"Produit le moins vendu : {produit_moins_vendu} ({unites_par_produit[produit_moins_vendu]} unités)")

Unités vendues par produit : {'Produit A': 1750, 'Produit B': 1055, 'Produit C': 575}
Produit le plus vendu  : Produit A (1750 unités)
Produit le moins vendu : Produit C (575 unités)


## 3. Graphiques avec Plotly Express

L'exemple fourni avec le projet (`app.py`) génère, avec Plotly Express, un camembert de la
quantité vendue par région et l'exporte en page HTML :

```python
figure = px.pie(données, values='qte', names='region', title='quantité vendue par région')
figure.write_html('ventes-par-region.html')
```

Sur le même principe, on crée deux nouveaux graphiques. Un **diagramme en barres** est
préféré au camembert : il permet de comparer directement les produits et d'afficher la valeur
exacte de chacun. Chaque graphique est affiché dans le notebook et exporté en page HTML dans le
dossier `graphiques/`.

In [ ]:
from pathlib import Path

# Totaux par produit : unités vendues et chiffre d'affaires
totaux_par_produit = ventes.groupby("produit", as_index=False)[["qte", "chiffre_affaires"]].sum()

# Libellé de chaque mesure, utilisé pour le titre et l'axe vertical des graphiques
LIBELLES = {"qte": "Unités vendues", "chiffre_affaires": "Chiffre d'affaires"}

DOSSIER_GRAPHIQUES = Path("graphiques")
DOSSIER_GRAPHIQUES.mkdir(exist_ok=True)


def creer_graphique_barres(colonne, fichier_html):
    """Diagramme en barres d'une mesure par produit, exporté en HTML puis renvoyé pour affichage."""
    libelle = LIBELLES[colonne]
    figure = px.bar(
        totaux_par_produit.sort_values(colonne, ascending=False),
        x="produit",
        y=colonne,
        title=f"{libelle} par produit",
        labels={"produit": "Produit", colonne: libelle},
        text=colonne,  # valeur affichée au sommet de chaque barre
        template="plotly_white",
    )
    figure.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
    figure.update_layout(separators=", ", yaxis_tickformat=",.0f")  # nombres au format français : 17 500
    figure.write_html(DOSSIER_GRAPHIQUES / fichier_html, include_plotlyjs="cdn")
    return figure

### 3.a Ventes par produit (unités vendues)

In [1]:
creer_graphique_barres("qte", "ventes-par-produit.html")

NameError: name 'creer_graphique_barres' is not defined

### 3.b Chiffre d'affaires par produit

In [8]:
creer_graphique_barres("chiffre_affaires", "ca-par-produit.html")

## Synthèse

Le chiffre d'affaires total et les totaux par produit ; les ventes par région sont couvertes
par la requête c de `requetes.sql`.

In [9]:
print(f"Chiffre d'affaires total : {ventes['chiffre_affaires'].sum():.0f}")
totaux_par_produit

Chiffre d'affaires total : 44825


,produit,qte,chiffre_affaires
0,Produit A,1750,17500
1,Produit B,1055,15825
2,Produit C,575,11500
